# Исследования развития игровой индустрии

- Автор: Федорова Марина Вячеславовна
- Дата: 04.12.2025

### Цели и задачи проекта

Используем датасет /datasets/new_games.csv и изучим развитие игровой индустрии с 2000 по 2013 года. Будем работать с данными взятыми из открытых источников. Данные датасета хранят информацию о продажах игр, выпущенных на разных платформах и в разных жанрах.

### Описание данных

В проекте будут использованы данные датасета `/datasets/new_games.csv` с таким описанием:
* `Name` — название игры.
* `Platform` — название платформы.
* `Year of Release` — год выпуска игры.
* `Genre` — жанр игры.
* `NA sales` — продажи в Северной Америке (в миллионах проданных копий).
* `EU sales` — продажи в Европе (в миллионах проданных копий).
* `JP sales` — продажи в Японии (в миллионах проданных копий).
* `Other sales` — продажи в других странах (в миллионах проданных копий).
* `Critic Score` — оценка критиков (от 0 до 100).
* `User Score` — оценка пользователей (от 0 до 10).
* `Rating` — рейтинг организации ESRB (англ. Entertainment Software Rating Board). Эта ассоциация определяет рейтинг компьютерных игр и присваивает им подходящую возрастную категорию.

<a class="ancor" id="10-bullet"></a>
### Содержание проекта

1. [Знакомство с данными.](#1-bullet) 
2. [Проверка ошибок в данных и предобработка.](#2-bullet) 
    * [Перевод названий столбцов в единый стиль](#3-bullet)
    * [Проверка типов данных.](#4-bullet)
    * [Поиск пропущенных значений.](#5-bullet)
    * [Поиск и обработка дубликатов.](#6-bullet)
    
    
3. [Фильтрация данных.](#7-bullet)
4. [Категоризация данных.](#8-bullet)
5. [Итоговый вывод.](#9-bullet)


<a class="ancor" id="1-bullet"></a>
## 1. Загрузка данных и знакомство с ними


Загрузим необходимые библиотеки для анализа данных и данные из датасета `/datasets/new_games.csv`.


In [1]:
# Импортируем библиотеку pandas
import pandas as pd

In [2]:
#Выгружаем данные из датасета /datasets/new_games.csv в датафрейм new_games
new_games = pd.read_csv('https://code.s3.yandex.net/datasets/new_games.csv')

Выведем информацию о данных с помощью метода `info()` и первые строки датафрейма `new_games`.


In [3]:
# Выводим информацию о датафрейме
new_games.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  object 
 1   Platform         16956 non-null  object 
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  object 
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  object 
 6   JP sales         16956 non-null  object 
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  object 
 10  Rating           10085 non-null  object 
dtypes: float64(4), object(7)
memory usage: 1.4+ MB


In [4]:
# Количество строк в датафрейме
row_cnt = new_games.shape[0]

In [5]:
# Выводим первые строки датафрейма на экран
new_games.head()

,Name,Platform,Year of Release,Genre,NA sales,EU sales,JP sales,Other sales,Critic Score,User Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


Датасет new_games содержит 11 столбцов и 16956 строк. 7 столбцов с типом данных `object` и 4 столбца с типом данных `float64`.

Изучим более подробно типы данных и их корректность:
* **Строковые значения (object).** В датасете представлено 7 столбцов с типом данных `object`: `Name`, `Platform`, `Genre`, `EU sales`, `JP sales`, `User Score`, `Rating`: 
    * `Name`, `Rating` содержат текстовую информацию (название игры, название платформы, жанр и рейтинг организации ESRB) и выбор типа данных `object` для этих данных является логичным.
    * `Platform`, `Genre` также содержат текстовую информацию, но их данные можно привести к типу данных `category`, т.к. название платформы и жанр игр можно рассматривать как категориальные данные.
    * `EU sales`, `JP sales` и `User Score` хранят информации о количестве продаж в Европе и Японии и об оценке пользователей. Так как данные представленны вещественными числами, разумнее было бы привести данные столбцы к типу данных `float64`.
* **Числовые значения с плавающей точкой (float64).** 4 столбца имеют тип данных `float64`: `Year of Release`, `NA sales`, `Other sales` и `Critic Score`, которые содержат информации о том в каком годы была выпущена игра, о продажах в Северной Америке, продажах в других странах и об оценке критиков. Для этих столбцов тип данных выбран корректно.

По данным выгрузки с помощью метода `info()`, шесть столбцов имеют пропущенные значения. Больше всего пропусков в столбце `User Score`. Вероятно, не все пользователи оценивали игры. 

Названия столбцов написаны в разных стилях и для удобства и дальнейшего анализа предпочтительно привести их к единому стилю.

[Назад к содержанию](#10-bullet)

---
<a class="ancor" id="2-bullet"></a>
## 2.  Проверка ошибок в данных и их предобработка
---
<a class="ancor" id="3-bullet"></a>
### 2.1. Названия, или метки, столбцов датафрейма

Выведим на экран названия всех столбцов датафрейма и проверим их стиль написания. При необходимости приведём все столбцы к стилю snake case.


In [6]:
# Проверим стиль написания столбцов, для этого выведим названия столбцов
new_games.columns

Index(['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales',
       'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating'],
      dtype='object')

In [7]:
# Приводим названия столбцов к стилю snace case
columns_snace_case = ['name', 'platform', 'year_of_release', 'genre', 'na_sales', 'eu_sales',
                      'jp_sales', 'other_sales', 'critic_score', 'user_score', 'rating']
new_games.columns = columns_snace_case

In [8]:
# Выводим названия столбцов для проверки на единый стиль
new_games.columns

Index(['name', 'platform', 'year_of_release', 'genre', 'na_sales', 'eu_sales',
       'jp_sales', 'other_sales', 'critic_score', 'user_score', 'rating'],
      dtype='object')

[Назад к содержанию](#10-bullet)

---
<a class="ancor" id="4-bullet"></a>
### 2.2. Типы данных

В данных встречаются столбцы с некорректным типом данных, которые необходимо преобразовать. Для начала заменим несоответствующие строковые значения, которые помешают преобразовать столбцы в числа - значением NaN.

In [9]:
# Заменим значения, не позволяющие преобразовать данные в числа, на NaN
new_games['eu_sales'] = new_games['eu_sales'].replace({'unknown':'NaN', 'tbd':'NaN'})
new_games['jp_sales'] = new_games['jp_sales'].replace({'unknown':'NaN', 'tbd':'NaN'})
new_games['user_score'] = new_games['user_score'].replace({'unknown':'NaN', 'tbd':'NaN'})

In [10]:
# Преобразуем строковые значения в тип float64
new_games[['eu_sales', 'jp_sales', 'user_score']] = new_games[['eu_sales', 'jp_sales', 'user_score']].astype('float64')

In [11]:
# Преобразуем столбец с жанром в тип category
new_games[['platform', 'genre']] = new_games[['platform', 'genre']].astype('category')

In [12]:
# Проверим результат преобразования
new_games[['eu_sales', 'jp_sales', 'user_score', 'platform', 'genre']].dtypes

eu_sales       float64
jp_sales       float64
user_score     float64
platform      category
genre         category
dtype: object

In [13]:
# Проверим изменения после переименования столбцов и преобразования данных
new_games.head()

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8.0,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8.0,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


Для дальнейшего исследования и оптимизации работы были сделаны следующие преобразования:
* `eu_sales`, `jp_sales`, `user_score` - тип данных изменен с `object` на `float64`
* `platform`, `genre` - изменены с `object` на `category`.

[Назад к содержанию](#10-bullet)

---
<a class="ancor" id="5-bullet"></a>
### 2.3. Наличие пропусков в данных

Посчитаем количество пропусков в каждом столбце в абсолютных и относительных значениях.


In [14]:
# Выводим количество пропущенных значений в датафрейме
new_games.isna().sum()

name                  2
platform              0
year_of_release     275
genre                 2
na_sales              0
eu_sales              6
jp_sales              4
other_sales           0
critic_score       8714
user_score         9268
rating             6871
dtype: int64

In [15]:
# Подсчитаем процент строк с пропусками
new_games.isna().mean() * 100

name                0.011795
platform            0.000000
year_of_release     1.621845
genre               0.011795
na_sales            0.000000
eu_sales            0.035386
jp_sales            0.023590
other_sales         0.000000
critic_score       51.391838
user_score         54.659118
rating             40.522529
dtype: float64

Изучив данные о количестве пропусков в абсолютных и относительных значениях, можно сделать некоторые выводы: 
* Больше всего пропущенных значений в столбцах с оценками пользователей `user_score` и критиков `critic_score` (более 50 % данных отсутствуют), а также рейтинге `rating` (около 41 %). Вероятно, это связано с отсутствием интернета в конце XX века и тем, что игры выпущенные в более раннем периоде и в других странах, менее известны и популярны в США и не охвачены американской рейтинговой системой. Пропуски в `rating` оставим как есть, т.к. данные не понадобятся для анализа. 
* Пропуски в столбце `year_of_release` в количестве 275 значений, что составляет 1.6 %, т.к. информация взята из открытых источников, это можно объяснить отсутствием данных на определенные игры, либо наличие дубликатов. Их тоже оставим как есть.  
* Пропуски с информацией о продажах в Европе `eu_sales` и Японии `jp_sales` составляют менее 1 % и их можно заменить средним значением в зависимости от года выпуска и платформы.
* Пропуски в столбцах `name` и `genre` можно удалить.


Для каждого случая обработаем пропущенные значения:

* Пропуски в столбцах name и genre удалим;
* В столбцах eu_sales и jp_sales пропуски заменим на среднее значение в зависимости от названия платформы и года выхода игры;
* Пропущенные значения в столбцах critic_score и user_score заменим на значение-индикатор `-1` для каждого столбца.

In [16]:
# Выясним в какой строке пропуски
new_games.loc[new_games['name'].isna() == True]

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
661,NaN,GEN,1993.0,NaN,1.78,0.53,0.00,0.08,NaN,NaN,NaN
14439,NaN,GEN,1993.0,NaN,0.00,0.00,0.03,0.00,NaN,NaN,NaN


In [17]:
# Удалим пропуски в столбцах 'name' и 'genre'
new_games = new_games.dropna(subset=['name', 'genre'])

In [18]:
# Заменим пропуски на среднее значение в зависимости от жанра и названия платформы
def count_sales_games(row):
    if pd.isna(row['eu_sales']):
        copy_game = new_games[(new_games['platform'] == row['platform']) &
                         (new_games['genre'] == row['genre'])]
        return copy_game['eu_sales'].mean()
    else:
        return row['eu_sales']
new_games['eu_sales'] = new_games.apply(count_sales_games, axis=1)

In [19]:
# Заменим пропуски на среднее значение в зависимости от жанра и названия платформы
def count_sales_games(row):
    if pd.isna(row['jp_sales']):
        copy_game = new_games[(new_games['platform'] == row['platform']) &
                         (new_games['genre'] == row['genre'])]
        return copy_game['jp_sales'].mean()
    else:
        return row['jp_sales']
new_games['jp_sales'] = new_games.apply(count_sales_games, axis=1)

In [20]:
# Заменим пропуски в столбце 'critic_score' на значение-индикатор
new_games['critic_score'] = new_games['critic_score'].fillna(-1)

In [21]:
# Заменим пропуски в столбце 'user_score' на значение-индикатор
new_games['user_score'] = new_games['user_score'].fillna(-1)

[Назад к содержанию](#10-bullet)

---
<a class="ancor" id="6-bullet"></a>
### 2.4. Явные и неявные дубликаты в данных

При необходимости названия или жанры игр можно приведем к нижнему регистру, а названия рейтинга — к верхнему. Изучим уникальные значения в категориальных данных. Проверим данные на неявные дубликаты, связанные с опечатками или разным способом написания и на явные дубликаты.

In [22]:
# Переводим строки столбца 'platform' в нижний регистр
new_games['platform'] = new_games['platform'].str.lower()

In [23]:
# Переводим строки столбца 'genre' в нижний регистр
new_games['genre'] = new_games['genre'].str.lower()

In [24]:
# Переводим названия игр в нижний регистр
new_games['name'] = new_games['name'].str.lower()

In [25]:
# Переводим строки столбца 'raiting' в верхний регистр
new_games['rating'] = new_games['rating'].str.upper()

In [26]:
# Находим список уникальных жанров 
new_games['genre'].unique()

array(['sports', 'platform', 'racing', 'role-playing', 'puzzle', 'misc',
       'shooter', 'simulation', 'action', 'fighting', 'adventure',
       'strategy'], dtype=object)

In [27]:
# Находим количесство уникальных жанров
new_games['genre'].nunique()

12

In [28]:
# Находим список уникальных названий платформ
new_games['platform'].unique()

array(['wii', 'nes', 'gb', 'ds', 'x360', 'ps3', 'ps2', 'snes', 'gba',
       'ps4', '3ds', 'n64', 'ps', 'xb', 'pc', '2600', 'psp', 'xone',
       'wiiu', 'gc', 'gen', 'dc', 'psv', 'sat', 'scd', 'ws', 'ng', 'tg16',
       '3do', 'gg', 'pcfx'], dtype=object)

In [29]:
# Находим количество уникальных платформ
new_games['platform'].nunique()

31

In [30]:
# Находим список уникальных годов
new_games['year_of_release'].unique()

array([2006., 1985., 2008., 2009., 1996., 1989., 1984., 2005., 1999.,
       2007., 2010., 2013., 2004., 1990., 1988., 2002., 2001., 2011.,
       1998., 2015., 2012., 2014., 1992., 1997., 1993., 1994., 1982.,
       2016., 2003., 1986., 2000.,   nan, 1995., 1991., 1981., 1987.,
       1980., 1983.])

In [31]:
# Находим количество уникальных годов
new_games['year_of_release'].nunique()

37

In [32]:
# Находим список уникальных рейтингов
new_games['rating'].unique()

array(['E', nan, 'M', 'T', 'E10+', 'K-A', 'AO', 'EC', 'RP'], dtype=object)

In [33]:
# Находим количество уникальных рейтингов
new_games['rating'].nunique()

8

In [34]:
# Удалим лишние пропуски в столбце с названиями игр
new_games['name'] = new_games['name'].str.strip()

In [35]:
# Находим список уникальных игр
new_games['name'].unique()

array(['wii sports', 'super mario bros.', 'mario kart wii', ...,
       'woody woodpecker in crazy castle 5', 'lma manager 2007',
       'haitaka no psychedelica'], dtype=object)

In [36]:
# Находим количество уникальных названий игр
new_games['name'].nunique()

11559

Количество уникальных данных: 
- по названию платформы (`31` уникальное значение), 
- по жанру (`12` уникальных жанров), 
- по годам (`37` уникальных значений),
- по рейтингу (`8` уникальных значений),
- по названию игр (`11559` уникальных игр из 16954 строк). 

Явные дубликаты встречаются в столбце `name`, т.к., судя по разнице количества уникальных названий игр и общим количеством строк, названия игр часто повторяются. Жанр, название платформы и года могут повторяться в столбцах - игры могут выходить на одной и той же платформе, в один год и в одинаковом жанре. 

In [37]:
# Выводим список дубликатов названия игр
duplicates = new_games[new_games.duplicated(subset=['name', 'year_of_release', 'platform'], keep=False)]
display(duplicates)

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
267,batman: arkham asylum,ps3,2009.0,action,2.24,1.31,0.07,0.61,91.0,8.9,T
268,batman: arkham asylum,ps3,2009.0,action,2.24,1.31,0.07,0.61,91.0,8.9,T
367,james bond 007: agent under fire,ps2,2001.0,shooter,1.90,1.13,0.10,0.41,72.0,7.9,T
368,james bond 007: agent under fire,ps2,2001.0,shooter,1.90,1.13,0.10,0.41,72.0,7.9,T
606,madden nfl 13,ps3,2012.0,sports,2.11,0.22,0.00,0.23,83.0,5.5,E
...,...,...,...,...,...,...,...,...,...,...,...
16799,transformers: prime,wii,2012.0,action,0.00,0.01,0.00,0.00,-1.0,-1.0,NaN
16911,metal gear solid v: the definitive experience,xone,2016.0,action,0.01,0.00,0.00,0.00,-1.0,-1.0,M
16912,metal gear solid v: the definitive experience,xone,2016.0,action,0.01,0.00,0.00,0.00,-1.0,-1.0,M
16939,the longest 5 minutes,psv,2016.0,action,0.00,0.00,0.01,0.00,-1.0,-1.0,NaN


In [38]:
# Подсчитаем количество дубликатов
duplicates_cnt = duplicates.shape[0]
display(duplicates_cnt)

484

 Количество явных дубликатов в столбце `name` составляет `484`. Прежде чем удалить дубликаты, отсортируем датафрейм по всем столбцам.

In [39]:
# Количество строк до удаления дубликатов
new_games.shape[0]

16954

In [40]:
# Сортируем датафрейм по всем столбцам
new_games_sorted = new_games.sort_values(by=list(new_games.columns))

In [41]:
# Удаляем дубликаты в столбце 'name'
new_games = new_games_sorted.drop_duplicates(subset=['name', 'year_of_release', 'platform'], keep=False)

In [42]:
# Количество строк после удаления дубликатов
new_games.shape[0]

16470

В процессе подготовки удалили пропуски в столбцах `name` и `genre`, удалили строки с явными дубликатами названия игр в столбце `name`. Посчитаем количество удалённых строк в абсолютном и относительном значениях. Так как пропуски в столбцах `name` и `genre` находятся в одной строке, подсчет количества удалённых строк можно производить без вычета количества пропусков в одном из столбцов, например, учитывая только столбец `name`.

In [43]:
# Подсчитаем количество удаленных строк после обработки данных
row_cnt - new_games['name'].isna().sum() - new_games.shape[0]

486

In [44]:
# Подсчитаем долю удаленных строк после обработки данных
round((new_games['name'].isna().sum() + duplicates_cnt) / row_cnt, 2)


0.03

Количество удаленных строк в процессе предобработки данных составило `486` строк (`0.03 %` от изначального количества строк), включая удаление пропусков по двум столбцам. 

[Назад к содержанию](#10-bullet)

---
<a class="ancor" id="7-bullet"></a>
## 3. Фильтрация данных

Отберём данные по году выпуска игры c 2000 по 2013 года включительно. Сохраним новый срез данных в отдельном датафрейме `new_games_actual`.

In [45]:
# Фильтруем данные по годам с 2000 по 2013
new_games_actual = new_games.loc[(new_games['year_of_release'] >= 2000) & (new_games['year_of_release'] <= 2013)].copy()

In [46]:
# Выводим информацию о количестве строк в датафрейме
new_games_actual.shape[0]

12580

In [47]:
# Выводим первые десять строк
new_games_actual.head(10)

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
8460,.hack//g.u. vol.1//rebirth,ps2,2006.0,role-playing,0.00,0.00,0.17,0.00,-1.0,-1.0,NaN
7182,.hack//g.u. vol.2//reminisce,ps2,2006.0,role-playing,0.11,0.09,0.00,0.03,-1.0,-1.0,NaN
8719,.hack//g.u. vol.2//reminisce (jp sales),ps2,2006.0,role-playing,0.00,0.00,0.16,0.00,-1.0,-1.0,NaN
8410,.hack//g.u. vol.3//redemption,ps2,2007.0,role-playing,0.00,0.00,0.17,0.00,-1.0,-1.0,NaN
1575,.hack//infection part 1,ps2,2002.0,role-playing,0.49,0.38,0.26,0.13,75.0,8.5,T
9189,.hack//link,psp,2010.0,role-playing,0.00,0.00,0.14,0.00,-1.0,-1.0,NaN
3023,.hack//mutation part 2,ps2,2002.0,role-playing,0.23,0.18,0.20,0.06,76.0,8.9,T
4313,.hack//outbreak part 3,ps2,2002.0,role-playing,0.14,0.11,0.17,0.04,70.0,8.7,T
8098,.hack//quarantine part 4: the final chapter,ps2,2003.0,role-playing,0.09,0.07,0.00,0.02,-1.0,-1.0,NaN
14525,.hack: sekai no mukou ni + versus,ps3,2012.0,action,0.00,0.00,0.03,0.00,-1.0,-1.0,NaN


Выполнили фильтрацию данных по столбцу `year_of_release` с 2000 по 2013 года включительно. За период с 2000 по 2013 года вышло 8730 игр.

[Назад к содержанию](#10-bullet)

---
<a class="ancor" id="8-bullet"></a>
## 4. Категоризация данных
    
Проведем категоризацию данных по оценкам пользователей и критиков. Сгруппируем по выделенным категориям и выделим топ-7 платформ по количеству выпущенных игр.

Разделим все игры по оценкам пользователей на категории: 
* высокая оценка (от 8 до 10 включительно)
* средняя оценка (от 3 до 8, не включая правую границу интервала) 
* низкая оценка (от 0 до 3, не включая правую границу интервала).

In [48]:
# Используем функцию для категоризации столбца 'user_score'
def categorize_us(row):
    if row['user_score'] < 3:
        return "Низкая оценка"
    elif row['user_score'] < 8:
        return "Средняя оценка"
    else:
        return "Высокая оценка"

In [49]:
# Применение функции ко всем строкам
new_games_actual['category_us'] = new_games_actual.apply(categorize_us, axis=1)

In [50]:
# Выводим первые строки после категоризации
new_games_actual.head()

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating,category_us
8460,.hack//g.u. vol.1//rebirth,ps2,2006.0,role-playing,0.00,0.00,0.17,0.00,-1.0,-1.0,NaN,Низкая оценка
7182,.hack//g.u. vol.2//reminisce,ps2,2006.0,role-playing,0.11,0.09,0.00,0.03,-1.0,-1.0,NaN,Низкая оценка
8719,.hack//g.u. vol.2//reminisce (jp sales),ps2,2006.0,role-playing,0.00,0.00,0.16,0.00,-1.0,-1.0,NaN,Низкая оценка
8410,.hack//g.u. vol.3//redemption,ps2,2007.0,role-playing,0.00,0.00,0.17,0.00,-1.0,-1.0,NaN,Низкая оценка
1575,.hack//infection part 1,ps2,2002.0,role-playing,0.49,0.38,0.26,0.13,75.0,8.5,T,Высокая оценка


Разделим все игры по оценкам критиков на категории: 
* высокая оценка (от 80 до 100 включительно)
* средняя оценка (от 30 до 80, не включая правую границу интервала)
* низкая оценка (от 0 до 30, не включая правую границу интервала).

In [51]:
# Используем функцию для категоризации столбца 'critic_score'
def categorize_cs(row):
    if row['critic_score'] < 30:
        return "Низкая оценка"
    elif row['critic_score'] < 80:
        return "Средняя оценка"
    else:
        return "Высокая оценка"

In [52]:
# Применение функции ко всем строкам
new_games_actual['category_cs'] = new_games_actual.apply(categorize_cs, axis=1)

Проверим результат после категоризации

In [53]:
# Выводим первые строки после категоризации
new_games_actual.head()

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating,category_us,category_cs
8460,.hack//g.u. vol.1//rebirth,ps2,2006.0,role-playing,0.00,0.00,0.17,0.00,-1.0,-1.0,NaN,Низкая оценка,Низкая оценка
7182,.hack//g.u. vol.2//reminisce,ps2,2006.0,role-playing,0.11,0.09,0.00,0.03,-1.0,-1.0,NaN,Низкая оценка,Низкая оценка
8719,.hack//g.u. vol.2//reminisce (jp sales),ps2,2006.0,role-playing,0.00,0.00,0.16,0.00,-1.0,-1.0,NaN,Низкая оценка,Низкая оценка
8410,.hack//g.u. vol.3//redemption,ps2,2007.0,role-playing,0.00,0.00,0.17,0.00,-1.0,-1.0,NaN,Низкая оценка,Низкая оценка
1575,.hack//infection part 1,ps2,2002.0,role-playing,0.49,0.38,0.26,0.13,75.0,8.5,T,Высокая оценка,Средняя оценка


Сгруппируем данные по выделенным категориям и посчитаем количество игр в каждой категории.

In [54]:
# Группируем данные по оценке пользователей и считаем количество игр
grouped_us_category = new_games_actual.groupby('category_us', as_index=False)['name'].count()

In [55]:
# Выводим результат группировки
display(grouped_us_category)

,category_us,name
0,Высокая оценка,2265
1,Низкая оценка,6303
2,Средняя оценка,4012


In [56]:
# Группируем данные по оценке критиков и считаем количество игр
grouped_cs_category = new_games_actual.groupby('category_cs', as_index=False)['name'].count()

In [57]:
# Выводим результат группировки
display(grouped_cs_category)

,category_cs,name
0,Высокая оценка,1670
1,Низкая оценка,5566
2,Средняя оценка,5344


Выделим топ-7 платформ по количеству игр, выпущенных за весь актуальный период.

In [58]:
# Группируем игры по платформе и считаем количество
platform_name = new_games_actual.groupby('platform', as_index=False)['name'].count()

In [59]:
# Сортируем платформы по количеству игр
platform_name_sort = platform_name.sort_values(by='name', ascending=False)

In [60]:
# Выводим топ-7 платформ
platform_name_sort.head(7)

,platform,name
9,ps2,2100
2,ds,2094
14,wii,1256
12,psp,1161
17,x360,1104
10,ps3,1065
4,gba,796


Топ-7 платформ с большим количеством игр: `ds-2087`, `ps2-1533`, `psp-842`, `gba-766`, `pc-663`, `wii-637`, `ps3-612`. Платформа `ds` является безусловным лидером по количеству игр.

[Назад к содержанию](#10-bullet)

---
<a class="ancor" id="9-bullet"></a>
## 5. Итоговый вывод



Для проведения исследований по развитию игр в XXI веке загрузили необходимые библиотеки для анализа данных и данные из датасета `/datasets/new_games.csv`. Датасет содержит `11 столбцов` и `16956 строк`. `7 столбцов` с типом данных `object` и `4 столбца` с типом данных `float64`. 
В ходе проверки и предобработки данных, была проделана следующая работа:
* Привели названия столбцов к единому стилю `snace case`;
* Заменили несоответствующие строковые значения, которые мешали преобразовывать в числа. Изменили тип данных некоторых столбцов: `eu_sales`, `jp_sales, user_score` - тип данных изменен с `object` на `float64` и `platform`, `genre` - изменены с `object` на `category`;
* Для каждого случая обработали пропущенные значения: пропуски в столбцах `name` и `genre` удалили; в столбцах `eu_sales` и `jp_sales` пропуски заменили на среднее значение в зависимости от названия платформы и года выхода игры; пропущенные значения в столбцах `critic_score` и `user_score` заменили на значение-индикатор; в столбце `year_of_release` и `rating` пропущенные значения оставили как есть;
* Проверили и провели работу с явными и неявными дубликатами. Подсчитали количество уникальных значений для категориальных данных;
* Провели фильтрацию данных по году выпуска игры с 2000 по 2013 года для дальнешего анализа. Сохранили новый срез данных в отдельном датафрейме `new_games_actual`. За период `с 2000 по 2013` год вышло `8730 игр`;
* Провели категоризацию данных по оценкам пользователей и критиков. Сгруппировали по выделенным категориям и выделили топ-7 платформ по количеству выпущенных игр.
    
   В процессе подготовки удалили пропуски в столбцах `name` и `genre`, удалили строки с явными дубликатами названия игр в столбце `name`. Посчитаем количество удалённых строк в абсолютном и относительном значениях. 

In [61]:
new_games_actual.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 12580 entries, 8460 to 9260
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             12580 non-null  object 
 1   platform         12580 non-null  object 
 2   year_of_release  12580 non-null  float64
 3   genre            12580 non-null  object 
 4   na_sales         12580 non-null  float64
 5   eu_sales         12580 non-null  float64
 6   jp_sales         12580 non-null  float64
 7   other_sales      12580 non-null  float64
 8   critic_score     12580 non-null  float64
 9   user_score       12580 non-null  float64
 10  rating           8597 non-null   object 
 11  category_us      12580 non-null  object 
 12  category_cs      12580 non-null  object 
dtypes: float64(7), object(6)
memory usage: 1.3+ MB


После предобработки данных, фильтрации по временному периоду и категоризации данных, появилось два новых стобца(`category_us` и `category_cs`) с типом данных `object`, а строк стало меньше - `8730`.

По данным категоризации можно сделать вывод, что и пользователи, и критики чаще ставят среднюю оценку. Категория со средней оценкой значительно превышает с количеством игр остальные категории.
Также можно выделить тройку лидеров платформ по количеству выпущенных игр: `ds -2087`, `ps2 - 1533` и `psp 842`.

[Назад к содержанию](#10-bullet)